# Day 5: Silver Standardization (OMOP CDM v5.4 & Entity Linkage)
This notebook demonstrates probabilistic patient linkage using Splink and PySpark across disparate EHR records (Fellegi-Sunter).

In [ ]:
!pip install pyspark pyiceberg duckdb splink

In [ ]:
from pyspark.sql import SparkSession
from splink.spark.linker import SparkLinker

spark = SparkSession.builder.appName("Day05_Splink_PySpark").getOrCreate()

# Sample disparate EHR records for linkage
data = [
    (1, "John", "Doe", "1980-01-01", "New York"),
    (2, "Jon", "Doe", "1980-01-01", "New York"),
    (3, "Jane", "Smith", "1990-05-15", "Boston"),
    (4, "J", "Smith", "1990-05-15", "Boston")
]
columns = ["unique_id", "first_name", "last_name", "dob", "city"]
df = spark.createDataFrame(data, columns)

print("Raw EHR Records:")
df.show()

# Splink Configuration for probabilistic linkage
settings = {
    "link_type": "dedupe_only",
    "blocking_rules_to_generate_predictions": [
        "l.dob = r.dob"
    ],
    "comparisons": [
        {
            "output_column_name": "first_name",
            "comparison_levels": [
                {
                    "sql_condition": "first_name_l IS NULL OR first_name_r IS NULL",
                    "label_for_charts": "Null",
                    "is_null_level": True,
                },
                {
                    "sql_condition": "first_name_l = first_name_r",
                    "label_for_charts": "Exact match",
                    "m_probability": 0.9,
                    "u_probability": 0.1,
                },
                {
                    "sql_condition": "levenshtein(first_name_l, first_name_r) <= 2",
                    "label_for_charts": "Levenshtein <= 2",
                    "m_probability": 0.05,
                    "u_probability": 0.2,
                },
                {
                    "sql_condition": "ELSE",
                    "label_for_charts": "All other comparisons",
                    "m_probability": 0.05,
                    "u_probability": 0.7,
                }
            ]
        },
        {
            "output_column_name": "last_name",
            "comparison_levels": [
                {
                    "sql_condition": "last_name_l IS NULL OR last_name_r IS NULL",
                    "label_for_charts": "Null",
                    "is_null_level": True,
                },
                {
                    "sql_condition": "last_name_l = last_name_r",
                    "label_for_charts": "Exact match",
                    "m_probability": 0.9,
                    "u_probability": 0.1,
                },
                {
                    "sql_condition": "ELSE",
                    "label_for_charts": "All other comparisons",
                    "m_probability": 0.1,
                    "u_probability": 0.9,
                }
            ]
        }
    ],
    "retain_matching_columns": True,
    "retain_intermediate_calculation_columns": True
}

linker = SparkLinker(df, settings, spark=spark)

predictions = linker.predict(threshold_match_probability=0.8)
results_df = predictions.as_spark_dataframe()

print("\nLinkage Results (Match Probability > 0.8):")
results_df.select('unique_id_l', 'unique_id_r', 'match_probability').show()
